# Is That Difference Real? DeLong, McNemar and Paired Resampling

Wiki reference for [comparing model performance](https://ml-viz-ruby.vercel.app/wiki/comparing-model-performance).

**The idea in one sentence.** Two models evaluated on the same test set make correlated errors, so the variance of their difference carries a covariance term, and a test that keeps that term has far more power than one that throws it away.

**Why it matters.** "0.86 versus 0.84, ship the first one" is a decision made on one finite sample. The paired tests here say whether the gap survives resampling, and the difference between the paired and unpaired answer is routinely the difference between shipping and not.

*Colab tip: File → Save a copy in Drive before editing.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from scipy import stats
from sklearn.metrics import roc_auc_score

plt.style.use('dark_background')
rng = np.random.default_rng(0)

## 1 — From scratch: DeLong's test

Ten subjects, four cases and six controls, scored by **both** models. The pairing is the whole point: subject `s5` is a hard control for A and for B alike.

In [ ]:
label = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
A = np.array([0.90, 0.75, 0.60, 0.35, 0.80, 0.55, 0.40, 0.30, 0.20, 0.10])
B = np.array([0.70, 0.65, 0.45, 0.25, 0.85, 0.60, 0.50, 0.40, 0.30, 0.15])

print(f'AUC_A = {roc_auc_score(label, A):.4f}   AUC_B = {roc_auc_score(label, B):.4f}')
print(f'gap   = {roc_auc_score(label, A) - roc_auc_score(label, B):.4f}')

The AUC is a two-sample U-statistic, so it decomposes into **placement values**: one contribution per subject.

$$V_{10}(p_i) = \frac{1}{n}\sum_j \psi(p_i, q_j), \qquad V_{01}(q_j) = \frac{1}{m}\sum_i \psi(p_i, q_j), \qquad \psi(x,y) = \mathbb{1}[x>y] + \tfrac{1}{2}\mathbb{1}[x=y]$$

Both average to the AUC. Their covariances across models are what DeLong needs.

In [ ]:
def placements(score, label):
    """V10 (one per case) and V01 (one per control); both average to the AUC."""
    pos = score[label == 1][:, None]      # (m, 1)
    neg = score[label == 0][None, :]      # (1, n)
    psi = (pos > neg) + 0.5 * (pos == neg)   # (m, n)
    return psi.mean(axis=1), psi.mean(axis=0)

v10_A, v01_A = placements(A, label)
v10_B, v01_B = placements(B, label)

print('model A  V10 (cases)   :', np.round(v10_A, 4))
print('model A  V01 (controls):', np.round(v01_A, 4))
print('model B  V10 (cases)   :', np.round(v10_B, 4))
print('model B  V01 (controls):', np.round(v01_B, 4))
print(f'\nmean(V10_A) = {v10_A.mean():.4f} = mean(V01_A) = {v01_A.mean():.4f} = AUC_A')

In [ ]:
def delong_test(scores, label):
    """DeLong's test for k correlated AUCs. scores: list of score arrays."""
    m = int(label.sum())
    n = len(label) - m
    V10, V01, aucs = [], [], []
    for s in scores:
        a, b = placements(s, label)
        V10.append(a); V01.append(b); aucs.append(a.mean())
    aucs = np.array(aucs)

    # np.cov wants variables in rows; subtract the AUC, not the row mean —
    # they are equal here, but the U-statistic theory is stated about the AUC.
    S10 = np.cov(np.array(V10), ddof=1)
    S01 = np.cov(np.array(V01), ddof=1)
    S = S10 / m + S01 / n

    var = S[0, 0] + S[1, 1] - 2 * S[0, 1]
    z = (aucs[0] - aucs[1]) / np.sqrt(var)
    p = 2 * stats.norm.sf(abs(z))
    return dict(aucs=aucs, S10=S10, S01=S01, S=S, var=var,
                se=np.sqrt(var), z=z, p=p)

res = delong_test([A, B], label)
print('S10 =\n', np.round(res['S10'], 6))
print('S01 =\n', np.round(res['S01'], 6))
print('S   =\n', np.round(res['S'], 6))
corr = res['S'][0, 1] / np.sqrt(res['S'][0, 0] * res['S'][1, 1])
print(f"\ncorr(AUC_A, AUC_B) = {corr:.3f}")
print(f"gap {res['aucs'][0] - res['aucs'][1]:.4f}, se {res['se']:.4f}, "
      f"z = {res['z']:.3f}, p = {res['p']:.4f}")

### The whole point, in two lines

Keep the covariance term or drop it. Nothing else changes.

In [ ]:
S = res['S']
gap = res['aucs'][0] - res['aucs'][1]
se_paired = np.sqrt(S[0, 0] + S[1, 1] - 2 * S[0, 1])
se_naive  = np.sqrt(S[0, 0] + S[1, 1])

for name, se in [('paired (DeLong)', se_paired), ('treated as independent', se_naive)]:
    z = gap / se
    print(f'{name:>23}: se {se:.4f}  z {z:6.3f}  p {2 * stats.norm.sf(abs(z)):.4f}')
print(f'\nthe covariance term removes {2 * S[0, 1] / (S[0, 0] + S[1, 1]):.0%} of the variance')
print(f'95% CI for the gap: [{gap - 1.96 * se_paired:.3f}, {gap + 1.96 * se_paired:.3f}]')

## 2 — Cross-checks: permutation and the paired bootstrap

The z-test is asymptotic, and with four cases that is a stretch. Two assumption-free checks on the same data.

**Exact permutation.** Under the null the two models are exchangeable, so swapping their predictions within a subject changes nothing. With ten subjects all $2^{10}$ swaps are enumerable, so no sampling is needed at all.

In [ ]:
def auc_gap(a, b, label):
    return roc_auc_score(label, a) - roc_auc_score(label, b)

observed = auc_gap(A, B, label)
extreme = total = 0
for flips in product([0, 1], repeat=len(label)):
    f = np.array(flips, bool)
    a = np.where(f, B, A)          # swap the two models' scores for those subjects
    b = np.where(f, A, B)
    total += 1
    extreme += abs(auc_gap(a, b, label)) >= abs(observed) - 1e-12

print(f'exact permutation: {extreme}/{total} = {extreme / total:.4f}')

**Paired bootstrap.** Resample *subjects* with replacement, stratified by class, and score both models on the same resampled indices. This generalises to any metric at all: precision at k, F1, expected cost, NDCG.

In [ ]:
def paired_bootstrap(A, B, label, metric=auc_gap, n_boot=20_000, seed=0):
    r = np.random.default_rng(seed)
    pos = np.flatnonzero(label == 1)
    neg = np.flatnonzero(label == 0)
    out = np.empty(n_boot)
    for i in range(n_boot):
        idx = np.concatenate([r.choice(pos, pos.size), r.choice(neg, neg.size)])
        out[i] = metric(A[idx], B[idx], label[idx])   # SAME indices for both
    return out

boot = paired_bootstrap(A, B, label)
p_boot = 2 * min((boot <= 0).mean(), (boot >= 0).mean())
print(f'bootstrap se   : {boot.std(ddof=1):.4f}')
print(f'95% percentile : [{np.percentile(boot, 2.5):.4f}, {np.percentile(boot, 97.5):.4f}]')
print(f'bootstrap p    : {p_boot:.4f}')

In [ ]:
print(f"{'method':>34} {'se':>8} {'p':>8}")
print(f"{'DeLong (paired, asymptotic)':>34} {se_paired:>8.3f} {res['p']:>8.4f}")
print(f"{'exact permutation':>34} {'-':>8} {extreme / total:>8.4f}")
print(f"{'paired bootstrap':>34} {boot.std(ddof=1):>8.3f} {p_boot:>8.4f}")
print(f"{'treated as independent':>34} {se_naive:>8.3f} "
      f"{2 * stats.norm.sf(abs(gap / se_naive)):>8.4f}")

The first two agree. The bootstrap is conservative here: with four cases the resampled gap takes few distinct values and a noticeable share of resamples land exactly at zero, which inflates the two-sided tail. That is a small-sample artefact and it disappears as $n$ grows. The last row is the one that is simply wrong, and it is the one that gets reported.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(boot, bins=40, color='#6366f1', alpha=0.85)
ax.axvline(observed, color='#14b8a6', lw=2, label=f'observed gap {observed:.3f}')
ax.axvline(0, color='#f43f5e', lw=2, ls='--', label='no difference')
ax.set_xlabel('AUC(A) − AUC(B) on a paired resample')
ax.set_ylabel('resamples')
ax.set_title('Paired bootstrap of the AUC gap (20,000 resamples)')
ax.legend(); plt.tight_layout(); plt.show()

**What to notice.** The distribution is lumpy, not smooth: ten subjects admit only so many distinct resampled AUCs. The mass sitting exactly at zero is what makes the bootstrap p-value conservative on data this small.

## 3 — McNemar's test at a fixed threshold

Once a threshold is fixed, the comparison becomes a table of agreement, and the cells where both models agree carry no information about which is better. McNemar's test discards them and asks whether the discordant split is more one-sided than a coin flip.

In [ ]:
def mcnemar(b, c, correction=True):
    """b: A right & B wrong. c: A wrong & B right."""
    stat = (abs(b - c) - (1 if correction else 0)) ** 2 / (b + c)
    chi_p = stats.chi2.sf(stat, 1)
    exact_p = 2 * stats.binom.cdf(min(b, c), b + c, 0.5)
    return stat, chi_p, min(exact_p, 1.0)

b, c = 25, 10          # out of 400 examples; they agree on the other 365
stat, chi_p, exact_p = mcnemar(b, c)
print(f'chi2 (continuity corrected): {stat:.3f}, p = {chi_p:.4f}')
print(f'chi2 (uncorrected)         : {mcnemar(b, c, False)[0]:.3f}, '
      f'p = {mcnemar(b, c, False)[1]:.4f}')
print(f'exact binomial             : p = {exact_p:.4f}')
print(f'\nthe verdict rests on {b + c} examples, not 400')

In [ ]:
# scikit-learn has no mcnemar; statsmodels does. Cross-check by simulation.
r = np.random.default_rng(3)
sim = r.binomial(b + c, 0.5, 200_000)          # null: each discordant is a coin flip
mc_p = ((np.abs(sim - (b + c) / 2) >= abs(b - (b + c) / 2)).mean())
print(f'simulated null p = {mc_p:.4f}  vs exact binomial {exact_p:.4f}')
assert abs(mc_p - exact_p) < 0.005, 'simulation should match the exact test'

## 4 — Tradeoffs and when to use which

| Test | Metric it compares | Assumes | Use when |
|---|---|---|---|
| **DeLong** | AUC | asymptotic normality; decent $m$ and $n$ | comparing ranking quality on a shared test set |
| **McNemar** | accuracy at a fixed threshold | discordant pairs are exchangeable | the threshold is already chosen |
| **Paired bootstrap** | any metric | resampling approximates the sampling distribution | bespoke metrics, or you also want an interval |
| **Permutation** | any metric | exchangeability under the null | small samples, or you want an exact p-value |
| **Nadeau–Bengio / 5×2cv** | any CV-averaged metric | corrects for overlapping training folds | comparing across cross-validation folds |

**Failure modes**

1. **Unpaired tests on a shared test set**: inflates the variance and hides real improvements.
2. **Bootstrapping each model separately**: destroys the pairing you are trying to exploit.
3. **A naive paired t-test across CV folds**: folds share training data, so the variance estimate is biased downward and the false-positive rate is several times nominal.
4. **Testing repeatedly on the same test set**: after 200 comparisons the p-values are decoration. Correct with Holm or Benjamini–Hochberg, and keep a final set scored once.

In [ ]:
# Multiple comparisons: 20 identical models, alpha = 0.05.
r = np.random.default_rng(1)
p_values = np.sort(r.uniform(size=20))        # the null is true for all 20
m, alpha = 20, 0.05

holm = 0
for i, pi in enumerate(p_values):             # step down until one fails
    if pi <= alpha / (m - i):
        holm += 1
    else:
        break

bh_idx = [i for i, pi in enumerate(p_values) if pi <= alpha * (i + 1) / m]
bh = max(bh_idx) + 1 if bh_idx else 0

print(f'smallest p-value      : {p_values[0]:.4f}')
print(f'raw p < 0.05          : {(p_values < alpha).sum()} "winner(s)"')
print(f'Bonferroni (0.05/20)  : {(p_values < alpha / m).sum()}')
print(f'Holm (step-down)      : {holm}')
print(f'Benjamini-Hochberg    : {bh}')
print(chr(10) + 'all 20 models are identical by construction, so every rejection is false')

## 5 — ✏️ Your turn

### Exercise 1: the covariance term by hand

Given the structural matrix $S$ for two correlated AUCs, compute the standard error of the difference both ways, and the correlation between the two estimates.

In [ ]:
S_given = np.array([[6.70e-5, 4.84e-5],      # var(A), cov(A, B)
                    [4.84e-5, 6.70e-5]])     # cov(A, B), var(B)
gap_given = 0.021

def exercise_1(S, gap):
    se_paired = None   # TODO(you): sqrt(S00 + S11 - 2*S01)
    se_naive = None    # TODO(you): sqrt(S00 + S11), pairing thrown away
    corr = None        # TODO(you): S01 / sqrt(S00 * S11)
    z_paired = None    # TODO(you)
    return se_paired, se_naive, corr, z_paired

In [ ]:
sp, sn, co, zp = exercise_1(S_given, gap_given)
assert abs(sp - 0.0060992) < 1e-6, f'paired se should be 0.006099, got {sp}'
assert abs(sn - 0.0115758) < 1e-6, f'naive se should be 0.011576, got {sn}'
assert abs(co - 0.7223881) < 1e-6, f'correlation should be 0.7224, got {co}'
assert abs(zp - 3.4431) < 1e-3, f'paired z should be 3.443, got {zp}'
print(f'paired se {sp:.6f}  naive se {sn:.6f}  ratio {sn / sp:.2f}x')
print(f'paired: z {zp:.2f}, p {2 * stats.norm.sf(zp):.4f}')
print(f'naive : z {gap_given / sn:.2f}, p {2 * stats.norm.sf(gap_given / sn):.4f}')

<details>
<summary>Solution</summary>

```python
def exercise_1(S, gap):
    se_paired = np.sqrt(S[0, 0] + S[1, 1] - 2 * S[0, 1])
    se_naive = np.sqrt(S[0, 0] + S[1, 1])
    corr = S[0, 1] / np.sqrt(S[0, 0] * S[1, 1])
    z_paired = gap / se_paired
    return se_paired, se_naive, corr, z_paired
```

A correlation of 0.72 halves the standard error, taking z from 1.81 ($p = 0.070$, "no evidence") to 3.44 ($p = 0.0006$, decisive) on an identical 0.021 gap. The covariance term is not a refinement, it is routinely the whole result.
</details>

### Exercise 2: a paired bootstrap for precision at k

DeLong covers AUC and McNemar covers accuracy. Everything else needs resampling. Implement the paired bootstrap interval for the difference in **precision at k = 100** between two rankers.

In [ ]:
def precision_at_k(score, label, k=100):
    top = np.argsort(-score)[:k]
    return label[top].mean()

r = np.random.default_rng(5)
n = 2000
y = r.binomial(1, 0.1, n)
s_A = y * r.normal(1.3, 1, n) + (1 - y) * r.normal(0, 1, n)
s_B = y * r.normal(1.0, 1, n) + (1 - y) * r.normal(0, 1, n)

def exercise_2(s_A, s_B, y, k=100, n_boot=2000, seed=1):
    rr = np.random.default_rng(seed)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = None        # TODO(you): n indices drawn WITH replacement, one set
        diffs[i] = None   # TODO(you): precision@k of A minus B, on those indices
    return diffs

In [ ]:
diffs = exercise_2(s_A, s_B, y)
lo, hi = np.percentile(diffs, [2.5, 97.5])
point = precision_at_k(s_A, y) - precision_at_k(s_B, y)
assert diffs.shape == (2000,), 'expected 2000 bootstrap replicates'
assert lo < point < hi, 'the observed difference should sit inside its own interval'
assert abs(np.median(diffs) - point) < 0.08, 'the bootstrap should centre near the estimate'
print(f'precision@100: A = {precision_at_k(s_A, y):.3f}, B = {precision_at_k(s_B, y):.3f}')
print(f'difference {point:+.3f}, 95% CI [{lo:+.3f}, {hi:+.3f}]')
print('CI excludes zero:', lo > 0 or hi < 0)

<details>
<summary>Solution</summary>

```python
for i in range(n_boot):
    idx = rr.integers(0, len(y), len(y))    # one index set...
    diffs[i] = (precision_at_k(s_A[idx], y[idx], k)
                - precision_at_k(s_B[idx], y[idx], k))   # ...used for BOTH models
```

The single shared `idx` is the whole exercise. Draw two independent index sets and you are comparing two unrelated resamples of the data, which reintroduces exactly the variance the pairing was there to remove.
</details>

## 6 — Key takeaways

- **Pairing is the result, not a detail.** $\operatorname{var}(\hat{A} - \hat{B})$ carries $-2\operatorname{cov}$, and that term is typically large and positive.
- **DeLong for AUCs**, via placement values and the $S_{10}/m + S_{01}/n$ decomposition.
- **McNemar at a fixed threshold**, on the discordant cells only; use the exact binomial when $b + c < 25$.
- **Paired bootstrap or permutation for anything else**, always resampling examples once and scoring both models on the same draw.
- **Report an interval with the effect size**, correct for multiple comparisons, and keep a test set you score exactly once.

**Next:** [concordance and rank statistics](https://ml-viz-ruby.vercel.app/wiki/concordance-and-rank-statistics) for what the AUC is made of, and [decile analysis](https://ml-viz-ruby.vercel.app/wiki/decile-analysis-and-lift) for where the risk landed.